# Variable grouping

Collapses the high-cardinality attributes into the groups the path-specific effect
estimator consumes, attaches the name-based proxies from `01-3_name_sampler.ipynb`,
and writes the table that `3_causal_estimation/causal_effect.py` reads.

Run with `--with_state_region --with_new_name_group`, which is what the paper reports.


In [ ]:
import pandas as pd

JOBDIST = "/path/to/CausalFair/Resume/job-distribution"


## 1. Grouping maps


In [ ]:
# Education: the seven ACS PUMS categories collapse to three levels (paper, Appendix).
# ['High school graduate/GED', 'Some college (no degree)', "Bachelor's", "Master's", "Associate's degree", 'Professional degree', 'Doctorate']
edu_level_map = {
    "High school graduate/GED": "low",
    "Some college (no degree)": "low",
    "Bachelor's": "medium",
    "Associate's degree": "medium",
    "Master's": "high",
    "Professional degree": "high",
    "Doctorate": "high",
}

# States collapse to the four U.S. Census Bureau regions (paper, Appendix).
region_map = {

    # -------------------------
    # Region 1: Northeast
    # -------------------------
    "Connecticut": "Northeast",
    "Maine": "Northeast",
    "Massachusetts": "Northeast",
    "New Hampshire": "Northeast",
    "Rhode Island": "Northeast",
    "Vermont": "Northeast",
    "New Jersey": "Northeast",
    "New York": "Northeast",
    "Pennsylvania": "Northeast",

    # -------------------------
    # Region 2: Midwest
    # -------------------------
    "Illinois": "Midwest",
    "Indiana": "Midwest",
    "Michigan": "Midwest",
    "Ohio": "Midwest",
    "Wisconsin": "Midwest",
    "Iowa": "Midwest",
    "Kansas": "Midwest",
    "Minnesota": "Midwest",
    "Missouri": "Midwest",
    "Nebraska": "Midwest",
    "North Dakota": "Midwest",
    "South Dakota": "Midwest",

    # -------------------------
    # Region 3: South
    # -------------------------
    "Delaware": "South",
    "District of Columbia": "South",
    "Florida": "South",
    "Georgia": "South",
    "Maryland": "South",
    "North Carolina": "South",
    "South Carolina": "South",
    "Virginia": "South",
    "West Virginia": "South",
    "Alabama": "South",
    "Kentucky": "South",
    "Mississippi": "South",
    "Tennessee": "South",
    "Arkansas": "South",
    "Louisiana": "South",
    "Oklahoma": "South",
    "Texas": "South",

    # -------------------------
    # Region 4: West
    # -------------------------
    "Arizona": "West",
    "Colorado": "West",
    "Idaho": "West",
    "Montana": "West",
    "Nevada": "West",
    "New Mexico": "West",
    "Utah": "West",
    "Wyoming": "West",
    "Alaska": "West",
    "California": "West",
    "Hawaii": "West",
    "Oregon": "West",
    "Washington": "West",
}


## 2. Apply them


In [ ]:
df = pd.read_csv(f"{JOBDIST}/processed_job_data_0102_with_exp_pred_with_names.csv", low_memory=False)

df["edu_level_group"] = df["edu_level"].map(edu_level_map)
df["state_region"] = df["state_name"].map(region_map)

assert df["edu_level_group"].notna().all(), "unmapped edu_level"
assert df["state_region"].notna().all(), "unmapped state_name"
print(df["edu_level_group"].value_counts().to_dict())
print(df["state_region"].value_counts().to_dict())


## 3. Attach the name-based proxies


In [ ]:
# Name-based redlining proxies, produced by 01-3_name_sampler.ipynb.
first_name = pd.read_csv(f"{JOBDIST}/name_grouping_first_final.csv", low_memory=False)
surname = pd.read_csv(f"{JOBDIST}/name_grouping_surname_final.csv", low_memory=False)

first_name = (first_name.rename(columns={"name": "first_name"})
              [["first_name", "first_name_age_group", "first_name_sex"]]
              .drop_duplicates("first_name"))
surname = (surname.rename(columns={"surname": "last_name"})
           [["last_name", "surname_race_label"]]
           .drop_duplicates("last_name"))

df = df.merge(first_name, on="first_name", how="left").merge(surname, on="last_name", how="left")

assert df[["first_name_age_group", "first_name_sex", "surname_race_label"]].notna().all().all()
print(df["first_name_sex"].value_counts().to_dict())
print(df["surname_race_label"].value_counts().to_dict())


## 4. Deduplicate by id and save


In [ ]:
# One row per resume id. Where an id appears more than once, keep the most complete row.
before = len(df)
df = (df.assign(_na=df.isna().sum(axis=1))
        .sort_values("_na")
        .drop_duplicates("id")
        .drop(columns="_na"))
print(f"{before:,} rows -> {len(df):,} unique ids")

df.to_csv(f"{JOBDIST}/processed_job_data_0102_with_exp_pred_with_names_and_grouping.csv", index=False)
